### Testing how to use my proposed equation

In [1]:
import numpy as np
import pandas as pd

# import and concatenate shear stress data
shear_stress_2022 = pd.read_csv("field_data/shear_stress_2022.csv", parse_dates=["time"])
shear_stress_2023 = pd.read_csv("field_data/shear_stress_2023.csv", parse_dates=["time"])
shear_stress = pd.concat([shear_stress_2022, shear_stress_2023])
# sort by time just to be safe
shear_stress = shear_stress.sort_values("time")

def antecedent_average(
    events_csv,
    shear_stress,
    event_time_col="date",
    shear_time_col="time",
    shear_col="tau*"
    ):
    # read data
    events = pd.read_csv(events_csv, parse_dates=[event_time_col]) 
    events = events.sort_values(event_time_col).reset_index(drop=True) # sort just to be safe
    results = []

    for i in range(len(events) - 1):
        t_start = events.loc[i, event_time_col]
        t_end = events.loc[i + 1, event_time_col]
        # subset shear stress between events
        mask = (shear_stress[shear_time_col] > t_start) & (shear_stress[shear_time_col] < t_end)
        shear_between = shear_stress.loc[mask, shear_col]
        # time between events (in hours)
        delta_t_hours = (t_end - t_start).total_seconds() / 3600
        # mean shear stress between events
        mean_tau = shear_between.mean()

        results.append({
            "event_start": t_start,
            "event_end": t_end,
            "time_between_events_hours": delta_t_hours,
            "mean_shear_between_events": mean_tau,
            "n_timesteps": shear_between.size
        })
    return pd.DataFrame(results)

def antecedent_average_below_tauc(
    events_csv,
    shear_stress,
    event_time_col="date",
    tauc_col="tauc*",
    shear_time_col="time",
    shear_col="tau*"
):
    # read and sort events
    events = pd.read_csv(events_csv, parse_dates=[event_time_col])
    events = events.sort_values(event_time_col).reset_index(drop=True)
    results = []

    for i in range(len(events) - 1):
        t_start = events.loc[i, event_time_col]
        t_end = events.loc[i + 1, event_time_col]
        tauc = events.loc[i, tauc_col]

        # subset shear stress between events
        mask = (shear_stress[shear_time_col] > t_start) & \
            (shear_stress[shear_time_col] < t_end)
        shear_between = shear_stress.loc[mask, shear_col]

        # exclude values above critical shear stress
        shear_below_tauc = shear_between[shear_between <= tauc]

        delta_t_hours = (t_end - t_start).total_seconds() / 3600
        mean_tau_below = shear_below_tauc.mean()

        results.append({
            "event_start": t_start,
            "event_end": t_end,
            "tauc": tauc,
            "time_between_events_hours": delta_t_hours,
            "mean_shear_below_tauc": mean_tau_below,
            "n_timesteps_total": shear_between.size,
            "n_timesteps_below_tauc": shear_below_tauc.size
        })
    return pd.DataFrame(results)

def antecedent_cumulative_shear(
    events_csv,
    shear_stress,
    event_time_col="date",
    shear_time_col="time",
    shear_col="tau*"
):
    # read and sort events
    events = pd.read_csv(events_csv, parse_dates=[event_time_col])
    events = events.sort_values(event_time_col).reset_index(drop=True)
    # compute timestep in seconds
    dt_seconds = shear_stress[shear_time_col].diff().dt.total_seconds().median()
    results = []

    for i in range(len(events) - 1):
        t_start = events.loc[i, event_time_col]
        t_end = events.loc[i + 1, event_time_col]

        mask = (shear_stress[shear_time_col] > t_start) & \
                (shear_stress[shear_time_col] < t_end)
        shear_between = shear_stress.loc[mask, shear_col]

        cumulative_tau = (shear_between * dt_seconds).sum()
        delta_t_hours = (t_end - t_start).total_seconds() / 3600

        results.append({
            "event_start": t_start,
            "event_end": t_end,
            "time_between_events_hours": delta_t_hours,
            "cumulative_shear": cumulative_tau,
            "n_timesteps": shear_between.size
        })
    return pd.DataFrame(results)

## No history dependent tau

### 1. Interpolated Rising D50

In [2]:
events = "field_data/1_interpolated_rising_d50.csv"
antecedent_1 = antecedent_average(events_csv=events,shear_stress=shear_stress)
antecedent_2 = antecedent_average_below_tauc(events_csv=events,shear_stress=shear_stress)
antecedent_3 = antecedent_cumulative_shear(events_csv=events,shear_stress=shear_stress)

In [3]:
antecedent_1.to_csv("field_data/results/1_interpolated_rising_d50/antecedent_average_shear.csv", index=False)
antecedent_2.to_csv("field_data/results/1_interpolated_rising_d50/antecedent_average_below_tauc.csv", index=False)
antecedent_3.to_csv("field_data/results/1_interpolated_rising_d50/antecedent_cumulative_shear.csv", index=False)

### 2. Exceedance D50

In [4]:
events = "field_data/2_exceedance_d50.csv"
antecedent_1 = antecedent_average(events_csv=events,shear_stress=shear_stress)
antecedent_2 = antecedent_average_below_tauc(events_csv=events,shear_stress=shear_stress)
antecedent_3 = antecedent_cumulative_shear(events_csv=events,shear_stress=shear_stress)

In [5]:
antecedent_1.to_csv("field_data/results/2_exceedance_d50/antecedent_average_shear.csv", index=False)
antecedent_2.to_csv("field_data/results/2_exceedance_d50/antecedent_average_below_tauc.csv", index=False)
antecedent_3.to_csv("field_data/results/2_exceedance_d50/antecedent_cumulative_shear.csv", index=False)

### 3. Interpolated Falling D50

In [6]:
events = "field_data/3_interpolated_falling_d50.csv"
antecedent_1 = antecedent_average(events_csv=events,shear_stress=shear_stress)
antecedent_2 = antecedent_average_below_tauc(events_csv=events,shear_stress=shear_stress)
antecedent_3 = antecedent_cumulative_shear(events_csv=events,shear_stress=shear_stress)

In [7]:
antecedent_1.to_csv("field_data/results/3_interpolated_falling_d50/antecedent_average_shear.csv", index=False)
antecedent_2.to_csv("field_data/results/3_interpolated_falling_d50/antecedent_average_below_tauc.csv", index=False)
antecedent_3.to_csv("field_data/results/3_interpolated_falling_d50/antecedent_cumulative_shear.csv", index=False)

### 4. Shortfall D50

In [9]:
events = "field_data/4_shortfall_d50.csv"
antecedent_1 = antecedent_average(events_csv=events,shear_stress=shear_stress)
antecedent_2 = antecedent_average_below_tauc(events_csv=events,shear_stress=shear_stress)
antecedent_3 = antecedent_cumulative_shear(events_csv=events,shear_stress=shear_stress) 

In [11]:
antecedent_1.to_csv("field_data/results/4_shortfall_d50/antecedent_average_shear.csv", index=False)
antecedent_2.to_csv("field_data/results/4_shortfall_d50/antecedent_average_below_tauc.csv", index=False)
antecedent_3.to_csv("field_data/results/4_shortfall_d50/antecedent_cumulative_shear.csv", index=False)